In [1]:
from mplsoccer import Sbopen
import pandas as pd

parser = Sbopen()

In [2]:
competitions = parser.competition()
competitions.head()

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2024-09-28T01:57:35.846538,None,None,2024-09-28T01:57:35.846538
3,16,4,Europe,Champions League,male,False,False,2018/2019,2025-05-08T15:10:50.835274,2021-06-13T16:17:31.694,None,2025-05-08T15:10:50.835274
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882


In [3]:
wc = competitions[
(competitions["competition_name"]=="FIFA World Cup") &
(competitions['season_name'].isin(['2018','2022']))
]
wc

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
29,43,106,International,FIFA World Cup,male,False,True,2022,2024-12-16T10:15:11.055845,2024-12-16T10:21:13.710934,2024-12-16T10:21:13.710934,2024-12-16T10:15:11.055845
30,43,3,International,FIFA World Cup,male,False,True,2018,2024-06-12T07:38:19.345758,2021-06-13T16:17:31.694,None,2024-06-12T07:38:19.345758


In [4]:
all_matches=[]
all_shots=[]

In [5]:
for _, tournament in wc.iterrows():

    competition_id = tournament['competition_id']
    season_id = tournament['season_id']

    print(f"Loading season: {tournament['season_name']}")

    matches = parser.match(
        competition_id=competition_id,
        season_id=season_id
    )

    all_matches.append(matches)

    for match_id in matches['match_id']:

        try:
            events, related, freeze, tactics = parser.event(match_id)

            shots = events[
                events['type_name'] == 'Shot'
            ].copy()

            shots['season_name'] = tournament['season_name']

            all_shots.append(shots)

        except Exception as e:
            print(f"Error in match {match_id}: {e}")

Loading season: 2022
Loading season: 2018


In [6]:
shots_df = pd.concat(all_shots, ignore_index=True)

In [7]:
shots_df.shape

(3200, 89)

In [10]:
shots_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 89 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              3200 non-null   object 
 1   index                           3200 non-null   int64  
 2   period                          3200 non-null   int64  
 3   timestamp                       3200 non-null   object 
 4   minute                          3200 non-null   int64  
 5   second                          3200 non-null   int64  
 6   possession                      3200 non-null   int64  
 7   duration                        3200 non-null   float64
 8   match_id                        3200 non-null   int64  
 9   type_id                         3200 non-null   int64  
 10  type_name                       3200 non-null   object 
 11  possession_team_id              3200 non-null   int64  
 12  possession_team_name            32

In [9]:
wc[['competition_name', 'season_name']]

,competition_name,season_name
29,FIFA World Cup,2022
30,FIFA World Cup,2018


In [11]:
shots_df.to_csv(
    "../data/raw/world_cup_shots.csv",
    index=False
)